<a href="https://colab.research.google.com/github/Dcodinginsane/AI_Web_APP/blob/main/AI_Agent.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import pandas as pd

# Load the Excel file
file_path = '/content/AdventureWorks Sales.xlsx'  # Replace with your file path
excel_data = pd.read_excel(file_path, sheet_name=None)  # sheet_name=None loads all sheets

# Check the sheet names
print("Sheet names:", excel_data.keys())

# Access each sheet by iterating through the dictionary
for sheet_name, df in excel_data.items():
    print(f"\nSheet name: {sheet_name}")
    print(df.head())  # Display the first few rows of each sheet


Sheet names: dict_keys(['Sales Order_data', 'Sales Territory_data', 'Sales_data', 'Reseller_data', 'Date_data', 'Product_data', 'Customer_data'])

Sheet name: Sales Order_data
    Channel  SalesOrderLineKey Sales Order Sales Order Line
0  Reseller           43659001     SO43659      SO43659 - 1
1  Reseller           43659002     SO43659      SO43659 - 2
2  Reseller           43659003     SO43659      SO43659 - 3
3  Reseller           43659004     SO43659      SO43659 - 4
4  Reseller           43659005     SO43659      SO43659 - 5

Sheet name: Sales Territory_data
   SalesTerritoryKey     Region        Country          Group
0                  1  Northwest  United States  North America
1                  2  Northeast  United States  North America
2                  3    Central  United States  North America
3                  4  Southwest  United States  North America
4                  5  Southeast  United States  North America

Sheet name: Sales_data
   SalesOrderLineKey  ResellerKey 

In [ ]:
import os
import json
import pandas as pd
import google.generativeai as genai

# Assign your API key
gemini_api_key = ""
genai.configure(api_key=gemini_api_key)

# File paths
excel_file_path = r"E:\LLM-PROJECTS\Langchain\data\AdventureWorks_Sales.xlsx"
output_file_path = r"E:\LLM-PROJECTS\Langchain\data\sheet_embeddings.jsonl"

# Read all sheets into a dictionary of DataFrames
sheets = pd.read_excel(excel_file_path, sheet_name=None)

def embed_content(content, sheet_name):
    """Embed content for the given sheet, handling size limit by chunking."""
    embeddings = []
    chunk_size = 5000  # Adjust based on the 10,000-byte limit

    # Split content into chunks
    for i in range(0, len(content), chunk_size):
        chunk = content[i:i+chunk_size]
        try:
            result = genai.embed_content(
                model="models/text-embedding-004",
                content=chunk,
                task_type="retrieval_document",
                title=f"Embedding of sheet: {sheet_name} (chunk {i//chunk_size + 1})"
            )
            embeddings.append(result['embedding'])
        except Exception as e:
            print(f"Error embedding chunk {i//chunk_size + 1} for sheet {sheet_name}: {e}")

    return embeddings

# Store embeddings in a JSON Lines file
with open(output_file_path, 'w') as f:
    for sheet_name, df in sheets.items():
        sheet_text = df.to_string(index=False)
        embeddings = embed_content(sheet_text, sheet_name)

        if embeddings:
            # Save embeddings with sheet name
            sheet_data = {
                "sheet_name": sheet_name,
                "embeddings": embeddings
            }
            f.write(json.dumps(sheet_data) + "\n")
            print(f"Embedding saved for sheet: {sheet_name}")
        else:
            print(f"Failed to generate embedding for sheet: {sheet_name}")

print(f"\nEmbeddings stored successfully in {output_file_path}")

# Generate Power Query using the Chat Model
generation_config = {
    "temperature": 1,
    "top_p": 0.95,
    "top_k": 64,
    "max_output_tokens": 8192,
    "response_mime_type": "text/plain",
}

# Initialize the chat model
model = genai.GenerativeModel(
    model_name="gemini-1.5-flash",
    generation_config=generation_config,
)

# Prepare context for Power Query generation
with open(output_file_path, 'r') as f:
    for line in f:
        sheet_data = json.loads(line)
        sheet_name = sheet_data["sheet_name"]
        embeddings = sheet_data["embeddings"]

        # Start chat session with the properly formatted history
        chat_session = model.start_chat(
            history=[
                {"parts": [{"role": "system", "content": "You are an assistant generating Power Query based on data embeddings."}]},
                {"parts": [{"role": "user", "content": f"Generate Power Query code for the sheet '{sheet_name}' based on the following embeddings: {embeddings}"}]}
            ]
        )

        # Ask for Power Query code generation
        response = chat_session.send_message("Generate Power Query code based on embeddings.")
        print(f"\nPower Query for '{sheet_name}':\n{response.text}\n")
